# MCD-rPPG Dataset: Preprocessing, Chunking & ROI Extraction Showcase

## Overview
This notebook demonstrates the complete preprocessing pipeline for the MCD-rPPG dataset:

1. **Video Frame Analysis**: Verify frame counts and alignment with PPG sync files
2. **Chunking Strategy**: Split videos into 450-frame chunks with 150-frame overlap
3. **ROI Extraction**: Extract 9 facial regions using MediaPipe landmarks
4. **Data Saving**: Save processed chunks as NPZ files with ROIs, PPG signals, and vital signs
5. **Visualization**: Showcase results for different camera orientations

### Key Parameters:
- **Chunk Size**: 450 frames
- **Overlap**: 150 frames (last 150 of chunk N = first 150 of chunk N+1)
- **ROIs**: 9 regions (full_face, forehead, left_eye, right_eye, nose, mouth, chin, left_iris, right_iris)
- **Output**: NPZ files with ROI data, PPG signals (value + time deltas), and vital signs

### Jitter Reduction:
MediaPipe landmarks are smoothed using a moving average window to prevent frame-to-frame jitter in ROI masks.

In [ ]:
# Install required packages
!pip install imageio[ffmpeg] -q
!pip install mediapipe -q
!pip install scikit-video -q
!pip install opencv-python -q
!pip install numba -q
!pip install scipy -q

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches
import seaborn as sns
import imageio
import cv2
from IPython.display import display
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

plt.style.use('seaborn-v0_8')

# ============================================================================
# CONFIGURATION
# ============================================================================
DATASET_PATH = '/home/cristic/data/Bgeorge/mcd_rppg/snapshots/929fb19c5ff2b5c8ed64a7c3a123744346674e88/'
DB_PATH = os.path.join(DATASET_PATH, 'db.csv')
MEDIAPIPE_MODEL_PATH = '/home/cristic/face_landmarker.task'
OUTPUT_PATH = '/home/cristic/preprocessed_data'

# Chunking parameters
CHUNK_SIZE = 450  # Number of frames per chunk
OVERLAP_SIZE = 150  # Number of overlapping frames

# ROI Configuration
ROIS = {
    'full_face': list(range(468)),
    'forehead': [103, 104, 105, 332, 333, 334, 6, 7, 8, 9, 10],
    'left_eye': list(range(22, 53)),
    'right_eye': list(range(252, 283)),
    'nose': list(range(1, 21)) + list(range(195, 221)),
    'mouth': list(range(60, 81)) + list(range(290, 321)),
    'chin': list(range(150, 171)) + list(range(370, 391)),
    'left_iris': list(range(468, 473)),
    'right_iris': list(range(473, 478))
}

# Fixed ROI bounding box size (W x H)
ROI_BOX_SIZE = (24, 24)

# Smoothing parameters for jitter reduction
SMOOTHING_WINDOW = 5  # Moving average window size for landmark smoothing

print('=' * 80)
print('CONFIGURATION LOADED')
print('=' * 80)
print(f'ROIs: {list(ROIS.keys())}')
print(f'Chunk size: {CHUNK_SIZE}, Overlap: {OVERLAP_SIZE}')
print(f'ROI box size: {ROI_BOX_SIZE}')
print(f'Smoothing window: {SMOOTHING_WINDOW}')

## 1. Load Database and Prepare Metadata

In [ ]:
# Load the database
df = pd.read_csv(DB_PATH)
meta_df = df.copy()

# Add full paths
file_columns = ['ecg', 'video', 'meta', 'ppg_sync']
for col in file_columns:
    if col in meta_df.columns:
        meta_df[f'{col}_full'] = meta_df[col].apply(
            lambda x: os.path.join(DATASET_PATH, x) if not os.path.isabs(x) else x
        )

# Add extracted metadata
meta_df['subject_id'] = meta_df['patient_id']
meta_df['condition'] = meta_df['step']
meta_df['camera_type'] = meta_df['camera']
meta_df['view_type'] = meta_df['view']

print('=' * 80)
print('DATABASE STRUCTURE')
print('=' * 80)
print(f'Total entries: {len(meta_df)}')
print(f'Unique subjects: {meta_df["patient_id"].nunique()}')
print(f'Camera types: {meta_df["camera_type"].unique()}')
print(f'View types: {meta_df["view_type"].unique()}')
print(f'Conditions: {meta_df["condition"].unique()}')

## 2. Video Frame Analysis and PPG Sync Verification

In [ ]:
def count_video_frames_ffmpeg(video_path):
    """Count frames in a video using ffmpeg (more reliable for .avi files)."""
    try:
        reader = imageio.get_reader(video_path, 'ffmpeg')
        n_frames = reader.count_frames()
        reader.close()
        return n_frames
    except Exception as e:
        print(f'Error counting frames for {video_path}: {e}')
        return 0

def count_ppg_sync_rows(ppg_sync_path):
    """Count rows in a PPG sync file."""
    try:
        if ppg_sync_path.endswith('.npy'):
            data = np.load(ppg_sync_path)
        elif ppg_sync_path.endswith('.txt'):
            data = np.loadtxt(ppg_sync_path)
        else:
            data = np.load(ppg_sync_path)
        return len(data)
    except Exception as e:
        print(f'Error loading {ppg_sync_path}: {e}')
        return 0

# Analyze frame counts and PPG sync alignment
print('=' * 80)
print('VIDEO FRAME & PPG SYNC ALIGNMENT ANALYSIS')
print('=' * 80)

# Sample a few videos for analysis
sample_size = min(10, len(meta_df))
sample_videos = meta_df.dropna(subset=['video_full', 'ppg_sync_full']).sample(n=sample_size, random_state=42)

results = []
for idx, row in tqdm(sample_videos.iterrows(), total=len(sample_videos), desc='Analyzing videos'):
    video_path = row['video_full']
    ppg_sync_path = row['ppg_sync_full']
    
    video_exists = os.path.exists(video_path)
    ppg_exists = os.path.exists(ppg_sync_path)
    
    n_video_frames = 0
    n_ppg_rows = 0
    
    if video_exists:
        n_video_frames = count_video_frames_ffmpeg(video_path)
    
    if ppg_exists:
        n_ppg_rows = count_ppg_sync_rows(ppg_sync_path)
    
    match = 'MATCH' if n_video_frames == n_ppg_rows and n_video_frames > 0 else 'MISMATCH'
    
    results.append({
        'subject_id': row['patient_id'],
        'condition': row['condition'],
        'camera': row['camera_type'],
        'view': row['view_type'],
        'video_frames': n_video_frames,
        'ppg_rows': n_ppg_rows,
        'match': match,
        'video_exists': video_exists,
        'ppg_exists': ppg_exists
    })

# Display results
results_df = pd.DataFrame(results)
print('\nFrame Count Analysis:')
display(results_df)

# Summary statistics
print('\nSummary Statistics:')
print(f'Videos with matching frames: {(results_df["match"] == "MATCH").sum()} / {len(results_df)}')
print(f'Average video frames: {results_df["video_frames"].mean():.0f}')
print(f'Average PPG rows: {results_df["ppg_rows"].mean():.0f}')

# Check for systematic mismatches
mismatches = results_df[results_df['match'] == 'MISMATCH']
if len(mismatches) > 0:
    print(f'\nWARNING: {len(mismatches)} videos have frame count mismatches!')
    display(mismatches)
else:
    print('\nAll sampled videos have matching frame counts!')

## 3. MediaPipe Landmark Detection with Temporal Smoothing

In [ ]:
try:
    import mediapipe as mp
    from mediapipe.tasks import python
    from mediapipe.tasks.python import vision
    MEDIAPIPE_AVAILABLE = True
    print('MediaPipe available')
except ImportError as e:
    MEDIAPIPE_AVAILABLE = False
    print(f'MediaPipe not available: {e}')

class TemporalSmoother:
    """Apply temporal smoothing to landmarks to reduce jitter."""
    
    def __init__(self, window_size=5):
        self.window_size = window_size
        self.history = []
    
    def smooth(self, landmarks):
        """Apply moving average smoothing to landmarks."""
        self.history.append(landmarks.copy())
        
        if len(self.history) < self.window_size:
            # Not enough history, return as-is
            return landmarks
        
        # Apply moving average with decreasing weights
        weights = [(self.window_size - i) / self.window_size for i in range(len(self.history))]
        smoothed = np.zeros_like(landmarks)
        for i, w in enumerate(weights):
            smoothed += self.history[i] * w
        smoothed /= sum(weights)
        
        # Keep history size bounded
        if len(self.history) > self.window_size * 2:
            self.history.pop(0)
        
        return smoothed

class MediaPipeLandmarkDetector:
    """MediaPipe-based landmark detector with temporal smoothing."""
    
    def __init__(self, model_path, smoothing_window=5):
        self.model_path = model_path
        self.smoothing_window = smoothing_window
        self.smoother = TemporalSmoother(smoothing_window)
        self.detector = None
        self.frame_count = 0
        self.initialize_detector()
    
    def initialize_detector(self):
        if not MEDIAPIPE_AVAILABLE:
            raise RuntimeError('MediaPipe not available')
        
        base_options = python.BaseOptions(model_asset_path=self.model_path)
        base_options.running_mode = vision.RunningMode.VIDEO        options = vision.FaceLandmarkerOptions(
            base_options=base_options,
            output_face_blendshapes=True,
            output_facial_transformation_matrixes=True,
            num_faces=1
        )
        self.detector = vision.FaceLandmarker.create_from_options(options)
    
    def detect_landmarks(self, frame):
        """Detect landmarks in a single frame."""
        if frame.dtype != np.uint8:
            frame = (frame * 255).astype(np.uint8)
        
        mp_image = mp.Image(image_format=mp.ImageFormat.SRGB, data=frame)
        
        try:
            result = self.detector.detect_for_video(mp_image, self.frame_count * 1000)            self.frame_count += 1
            
            if result and result.face_landmarks:
                face_landmarks = result.face_landmarks[0]
                frame_width, frame_height = frame.shape[1], frame.shape[0]
                points = np.array([(lm.x * frame_width, lm.y * frame_height) for lm in face_landmarks], dtype='float32')
                # Apply temporal smoothing
                smoothed_points = self.smoother.smooth(points)
                return smoothed_points
            else:
                # No face detected
                return None
                
        except Exception as e:
            print(f'Error in landmark detection: {e}')
            return None
    
    def reset(self):
        """Reset the detector for a new video."""
        self.frame_count = 0
        self.smoother.history = []

# Initialize detector
if MEDIAPIPE_AVAILABLE:
    detector = MediaPipeLandmarkDetector(MEDIAPIPE_MODEL_PATH, smoothing_window=SMOOTHING_WINDOW)
    print(f'MediaPipe detector initialized with smoothing window = {SMOOTHING_WINDOW}')
else:
    print('MediaPipe not available - landmark detection will be skipped')

## 4. Chunking Strategy Implementation

In [ ]:
def create_chunks(n_frames, chunk_size=450, overlap_size=150):
    """
    Create chunk indices for a video with given frame count.
    
    Args:
        n_frames: Total number of frames in the video
        chunk_size: Number of frames per chunk
        overlap_size: Number of overlapping frames between chunks
    
    Returns:
        List of chunk tuples: (start_frame, end_frame, chunk_index)
    """
    chunks = []
    chunk_idx = 0
    start = 0
    
    while start < n_frames:
        end = min(start + chunk_size, n_frames)
        chunks.append((start, end, chunk_idx))
        
        # Move to next chunk with overlap
        if end == n_frames:
            break
        start = end - overlap_size
        chunk_idx += 1
    
    return chunks

def get_chunk_info(chunks):
    """Get information about chunks."""
    info = []
    for start, end, idx in chunks:
        info.append({
            'chunk_index': idx,
            'start_frame': start,
            'end_frame': end,
            'num_frames': end - start,
            'overlap_with_next': min(OVERLAP_SIZE, end - start) if idx < len(chunks) - 1 else 0
        })
    return pd.DataFrame(info)

# Test chunking on sample frame counts
print('=' * 80)
print('CHUNKING STRATEGY TEST')
print('=' * 80)

test_frame_counts = [450, 900, 1350, 1800, 2250, 4500]

for n_frames in test_frame_counts:
    chunks = create_chunks(n_frames, CHUNK_SIZE, OVERLAP_SIZE)
    chunk_info = get_chunk_info(chunks)
    
    print(f'Video with {n_frames} frames:')
    print(f'  Number of chunks: {len(chunks)}')
    print(f'  Chunk sizes: {chunk_info["num_frames"].tolist()}')
    print(f'  Total frames covered: {chunk_info["num_frames"].sum()}')
    
    # Verify overlap
    if len(chunks) > 1:
        for i in range(len(chunks) - 1):
            start1, end1, _ = chunks[i]
            start2, end2, _ = chunks[i + 1]
            overlap = end1 - start2
            print(f'  Overlap between chunk {i} and {i+1}: {overlap} frames')

# Visualize chunking for a sample video
sample_n_frames = 1800
chunks = create_chunks(sample_n_frames, CHUNK_SIZE, OVERLAP_SIZE)
chunk_info = get_chunk_info(chunks)

plt.figure(figsize=(15, 3))
for _, row in chunk_info.iterrows():
    plt.axvspan(row['start_frame'], row['end_frame'], alpha=0.3, 
               label=f'Chunk {row["chunk_index"]}' if row['chunk_index'] == 0 else '')
plt.xlabel('Frame Index')
plt.ylabel('Chunk')
plt.title(f'Chunking Strategy for {sample_n_frames}-frame Video')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. ROI Extraction with Fixed Bounding Boxes

In [ ]:
def extract_roi_bbox(landmarks, roi_indices, frame_shape, box_size=(24, 24)):    """
    Extract bounding box for a specific ROI.
    
    Args:
        landmarks: Landmarks array of shape (468, 2)
        roi_indices: List of landmark indices for the ROI
        frame_shape: Shape of the frame (H, W)
        box_size: Fixed size of the bounding box (W, H)
    
    Returns:
        Bounding box as (x, y, w, h) where (x, y) is the top-left corner
    """
    # Get ROI landmarks
    valid_indices = [i for i in roi_indices if i < len(landmarks)]
    if not valid_indices:
        return (0, 0, box_size[0], box_size[1])
    
    roi_points = landmarks[valid_indices]
    
    # Calculate center
    center_x = int(np.mean(roi_points[:, 0]))
    center_y = int(np.mean(roi_points[:, 1]))
    
    # Calculate bounding box (centered on ROI)
    box_w, box_h = box_size
    x = max(0, center_x - box_w // 2)
    y = max(0, center_y - box_h // 2)
    
    # Ensure box fits within frame
    x = min(x, frame_shape[1] - box_w)
    y = min(y, frame_shape[0] - box_h)
    
    return (x, y, box_w, box_h)

def extract_roi_region(frame, bbox):
    """Extract ROI region from a frame given bounding box."""
    x, y, w, h = bbox
    return frame[y:y+h, x:x+w]

def extract_all_rois_for_frame(frame, landmarks, rois_dict, box_size=(24, 24)):
    """Extract all ROIs for a single frame."""
    frame_shape = frame.shape[:2]
    roi_data = {}
    
    for roi_name, roi_indices in rois_dict.items():
        bbox = extract_roi_bbox(landmarks, roi_indices, frame_shape, box_size)
        roi_region = extract_roi_region(frame, bbox)
        roi_data[roi_name] = roi_region
    
    return roi_data

print('ROI extraction functions ready')

## 6. Complete Preprocessing Pipeline

In [ ]:
def load_ppg_sync_data(ppg_sync_path):
    """Load PPG sync data from file."""
    try:
        if ppg_sync_path.endswith('.npy'):
            data = np.load(ppg_sync_path)
        elif ppg_sync_path.endswith('.txt'):
            data = np.loadtxt(ppg_sync_path)
        else:
            data = np.load(ppg_sync_path)
        
        # Ensure 2D array
        if data.ndim == 1:
            data = data.reshape(-1, 1)
        
        return data
    except Exception as e:
        print(f'Error loading {ppg_sync_path}: {e}')
        return np.array([])

def load_video_frames(video_path, start_frame=0, end_frame=None):
    """Load video frames using ffmpeg."""
    try:
        reader = imageio.get_reader(video_path, 'ffmpeg')
        total_frames = reader.count_frames()
        
        if end_frame is None:
            end_frame = total_frames
        
        # Read specific frame range
        frames = []
        for i in range(start_frame, end_frame):
            frame = reader.get_next_data()
            frames.append(frame)
        
        reader.close()
        return np.array(frames)
    except Exception as e:
        print(f'Error loading video {video_path}: {e}')
        return np.array([])

def process_video_chunk(video_path, ppg_sync_path, meta_data, chunk_start, chunk_end, chunk_idx):
    """
    Process a single video chunk: extract frames, landmarks, ROIs, and PPG data.
    
    Args:
        video_path: Path to video file
        ppg_sync_path: Path to PPG sync file
        meta_data: Dictionary with metadata
        chunk_start: Start frame index
        chunk_end: End frame index
        chunk_idx: Chunk index
    
    Returns:
        Dictionary with processed chunk data
    """
    # Load video frames for this chunk
    video_frames = load_video_frames(video_path, chunk_start, chunk_end)
    
    if len(video_frames) == 0:
        print(f'  No frames loaded for chunk {chunk_idx}')
        return None
    
    # Load PPG sync data
    ppg_sync_data = load_ppg_sync_data(ppg_sync_path)
    
    if len(ppg_sync_data) == 0:
        print(f'  No PPG data loaded')
        return None
    
    # Extract PPG data for this chunk
    chunk_ppg = ppg_sync_data[chunk_start:chunk_end]
    
    # Separate PPG signal and time deltas
    if chunk_ppg.shape[1] >= 2:
        ppg_values = chunk_ppg[:, 0]
        time_deltas = chunk_ppg[:, 1]
    else:
        ppg_values = chunk_ppg[:, 0] if chunk_ppg.ndim > 1 else chunk_ppg
        time_deltas = np.zeros_like(ppg_values)
    
    # Detect landmarks for all frames in chunk
    detector.reset()
    chunk_landmarks = []
    
    for frame in video_frames:
        lms = detector.detect_landmarks(frame)
        if lms is not None:
            chunk_landmarks.append(lms)
        else:
            # Use previous landmarks
            if chunk_landmarks:
                chunk_landmarks.append(chunk_landmarks[-1].copy())
            else:
                print(f'  No landmarks in first frame of chunk {chunk_idx}')
                return None
    
    chunk_landmarks = np.array(chunk_landmarks)
    
    # Extract ROIs for all frames
    roi_data = {}
    for roi_name, roi_indices in ROIS.items():
        roi_frames = []
        for frame_idx, frame in enumerate(video_frames):
            landmarks = chunk_landmarks[frame_idx]
            bbox = extract_roi_bbox(landmarks, roi_indices, frame.shape[:2], ROI_BOX_SIZE)
            roi_region = extract_roi_region(frame, bbox)
            roi_frames.append(roi_region)
        roi_data[roi_name] = np.array(roi_frames)
    
    # Prepare chunk metadata
    chunk_meta = {
        'subject_id': meta_data.get('subject_id'),
        'condition': meta_data.get('condition'),
        'camera_type': meta_data.get('camera_type'),
        'view_type': meta_data.get('view_type'),
        'chunk_index': chunk_idx,
        'start_frame': chunk_start,
        'end_frame': chunk_end,
        'num_frames': chunk_end - chunk_start,
        'video_path': video_path,
        'ppg_sync_path': ppg_sync_path
    }
    
    # Get vital signs from metadata if available
    vital_signs = {}
    vital_cols = ['weight', 'height', 'bmi', 'age', 'upper_ap', 'lower_ap', 
                  'saturation', 'temperature', 'hemoglobin', 'glycated_hemoglobin', 
                  'cholesterol', 'respiratory', 'rigidity', 'pulse', 'stress']
    for col in vital_cols:
        if col in meta_data:
            vital_signs[col] = meta_data[col]
    
    return {
        'roi_data': roi_data,
        'ppg_values': ppg_values,
        'time_deltas': time_deltas,
        'landmarks': chunk_landmarks,
        'metadata': chunk_meta,
        'vital_signs': vital_signs
    }

def save_chunk_as_npz(chunk_data, output_path):
    """Save chunk data as NPZ file."""
    try:
        os.makedirs(output_path, exist_ok=True)
        
        # Prepare data for saving
        save_data = {}
        
        # Save ROI data
        for roi_name, roi_frames in chunk_data['roi_data'].items():
            save_data[f'roi_{roi_name}'] = roi_frames
        
        # Save PPG data
        save_data['ppg_values'] = chunk_data['ppg_values']
        save_data['time_deltas'] = chunk_data['time_deltas']
        
        # Save landmarks
        save_data['landmarks'] = chunk_data['landmarks']
        
        # Save metadata
        for key, value in chunk_data['metadata'].items():
            save_data[f'meta_{key}'] = value
        
        # Save vital signs
        for key, value in chunk_data['vital_signs'].items():
            save_data[f'vital_{key}'] = value
        
        # Generate filename
        subject_id = chunk_data['metadata']['subject_id']
        camera = chunk_data['metadata']['camera_type']
        condition = chunk_data['metadata']['condition']
        chunk_idx = chunk_data['metadata']['chunk_index']
        filename = f'{subject_id}_{camera}_{condition}_chunk{chunk_idx}.npz'
        filepath = os.path.join(output_path, filename)
        
        # Save as NPZ
        np.savez_compressed(filepath, **save_data)
        
        print(f'  Saved: {filepath}')
        return filepath
        
    except Exception as e:
        print(f'Error saving chunk: {e}')
        return None

def process_complete_video(video_row, output_path=OUTPUT_PATH):
    """
    Process a complete video: create chunks and save as NPZ files.
    
    Args:
        video_row: DataFrame row with video information
        output_path: Directory to save NPZ files
    
    Returns:
        List of saved NPZ file paths
    """
    video_path = video_row['video_full']
    ppg_sync_path = video_row['ppg_sync_full']
    
    print(f'Processing video: {os.path.basename(video_path)}')
    print(f'  Subject: {video_row["patient_id"]}, Condition: {video_row["condition"]}')
    print(f'  Camera: {video_row["camera_type"]}, View: {video_row["view_type"]}')
    
    # Count frames
    n_frames = count_video_frames_ffmpeg(video_path)
    print(f'  Total frames: {n_frames}')
    
    # Create chunks
    chunks = create_chunks(n_frames, CHUNK_SIZE, OVERLAP_SIZE)
    print(f'  Number of chunks: {len(chunks)}')
    
    # Prepare metadata
    meta_data = {
        'subject_id': video_row['patient_id'],
        'condition': video_row['condition'],
        'camera_type': video_row['camera_type'],
        'view_type': video_row['view_type']
    }
    
    # Add vital signs
    vital_cols = ['weight', 'height', 'bmi', 'age', 'upper_ap', 'lower_ap', 
                  'saturation', 'temperature', 'hemoglobin', 'glycated_hemoglobin', 
                  'cholesterol', 'respiratory', 'rigidity', 'pulse', 'stress']
    for col in vital_cols:
        if col in video_row:
            meta_data[col] = video_row[col]
    
    saved_files = []
    
    # Process each chunk
    for start, end, chunk_idx in tqdm(chunks, desc=f'Processing chunks for {os.path.basename(video_path)}'):
        chunk_data = process_video_chunk(
            video_path, ppg_sync_path, meta_data, start, end, chunk_idx
        )
        
        if chunk_data is not None:
            filepath = save_chunk_as_npz(chunk_data, output_path)
            if filepath:
                saved_files.append(filepath)
    
    return saved_files

print('Preprocessing pipeline functions ready')

## 7. Showcase: Process and Visualize One Video from Each Camera Orientation

In [ ]:
if MEDIAPIPE_AVAILABLE:
    print('=' * 80)
    print('SHOWCASE: PROCESSING ONE VIDEO FROM EACH CAMERA ORIENTATION')
    print('=' * 80)
    
    # Get one video from each camera type
    camera_types = meta_df['camera_type'].unique()
    showcase_videos = []
    
    for camera in camera_types:
        samples = meta_df[meta_df['camera_type'] == camera].dropna(subset=['video_full', 'ppg_sync_full'])
        if len(samples) > 0:
            # Pick the first one
            showcase_videos.append(samples.iloc[0])
    
    print(f'Found {len(showcase_videos)} camera types to showcase')
    
    # Process each showcase video
    showcase_results = []
    
    for video_row in showcase_videos:
        print(f'Processing {video_row["camera_type"]} camera')
        
        # For showcase, we'll process just the first chunk
        video_path = video_row['video_full']
        ppg_sync_path = video_row['ppg_sync_full']
        
        # Count frames
        n_frames = count_video_frames_ffmpeg(video_path)
        print(f'  Video frames: {n_frames}')
        
        # Create first chunk
        chunks = create_chunks(n_frames, CHUNK_SIZE, OVERLAP_SIZE)
        if chunks:
            start, end, chunk_idx = chunks[0]
            print(f'  Processing first chunk: frames {start}-{end}')
            
            # Prepare metadata
            meta_data = {
                'subject_id': video_row['patient_id'],
                'condition': video_row['condition'],
                'camera_type': video_row['camera_type'],
                'view_type': video_row['view_type']
            }
            
            # Add vital signs
            vital_cols = ['pulse', 'saturation', 'upper_ap', 'lower_ap']
            for col in vital_cols:
                if col in video_row:
                    meta_data[col] = video_row[col]
            
            # Process chunk
            chunk_data = process_video_chunk(
                video_path, ppg_sync_path, meta_data, start, end, chunk_idx
            )
            
            if chunk_data is not None:
                showcase_results.append(chunk_data)
                print(f'  Successfully processed {video_row["camera_type"]} camera')
            else:
                print(f'  Failed to process {video_row["camera_type"]} camera')
        else:
            print(f'  No chunks created for {video_row["camera_type"]}')
    
    print(f'Successfully processed {len(showcase_results)} camera orientations')
else:
    print('Skipping showcase - MediaPipe not available')

In [ ]:
import osimport matplotlib.pyplot as pltif MEDIAPIPE_AVAILABLE and 'showcase_results' in locals() and showcase_results:    print('=' * 80)    print('SHOWCASE VISUALIZATION')    print('=' * 80)    print()    for chunk_data in showcase_results:        metadata = chunk_data['metadata']        camera = metadata['camera_type']        subject = metadata['subject_id']        condition = metadata['condition']        n_frames = metadata['num_frames']        start_frame = metadata['start_frame']        end_frame = metadata['end_frame']        print(f'--- {camera} Camera (Subject {subject}, {condition}) ---')        print(f'Chunk: frames {start_frame}-{end_frame} ({n_frames} frames)')        print()        vital_signs = chunk_data['vital_signs']        if vital_signs:            print('Vital Signs:')            for key, value in vital_signs.items():                print(f'  {key}: {value}')        print()        print('NPZ File Structure:')        print('-' * 40)        for roi_name, roi_frames in chunk_data['roi_data'].items():            print(f'  roi_{roi_name}: shape {roi_frames.shape}')        print(f'  ppg_values: shape {chunk_data["ppg_values"].shape}')        print(f'  time_deltas: shape {chunk_data["time_deltas"].shape}')        print(f'  landmarks: shape {chunk_data["landmarks"].shape}')        for key, value in metadata.items():            print(f'  meta_{key}: {value}')        for key, value in vital_signs.items():            print(f'  vital_{key}: {value}')        print()        video_frames = chunk_data['roi_data']['full_face']        frame_indices = [0, min(n_frames // 2, n_frames - 1)]        roi_items = list(chunk_data['roi_data'].items())        n_rois = len(roi_items)        fig, axes = plt.subplots(2, n_rois + 1, figsize=(20, 8))        for row_idx, frame_idx in enumerate(frame_indices):            axes[row_idx, 0].imshow(video_frames[frame_idx])            axes[row_idx, 0].set_title(f'Frame {start_frame + frame_idx}' if row_idx == 0 else '')            axes[row_idx, 0].axis('off')            for col_idx, (roi_name, roi_frames) in enumerate(roi_items, start=1):                axes[row_idx, col_idx].imshow(roi_frames[frame_idx])                axes[row_idx, col_idx].set_title(roi_name if row_idx == 0 else '')                axes[row_idx, col_idx].axis('off')        plt.suptitle(f'{camera} Camera - ROI Extraction Showcase', fontsize=14, y=0.98)        plt.tight_layout()        plt.show()        print()        ppg_values = chunk_data['ppg_values']        time_deltas = chunk_data['time_deltas']        fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(15, 8))        ax1.plot(ppg_values, color='red', linewidth=1.5)        ax1.set_title(f'PPG Signal - {camera} Camera | Frames {start_frame}-{end_frame}')        ax1.set_xlabel('Frame Index (within chunk)')        ax1.set_ylabel('PPG Value')        ax1.grid(True, alpha=0.3)        ax2.plot(time_deltas, color='blue', linewidth=1.5)        ax2.set_title('Time Deltas (Frame-to-Frame Intervals)')        ax2.set_xlabel('Frame Index (within chunk)')        ax2.set_ylabel('Time Delta (seconds)')        ax2.grid(True, alpha=0.3)        plt.tight_layout()        plt.show()        print()        save_path = os.path.join(OUTPUT_PATH, 'showcase')        os.makedirs(save_path, exist_ok=True)        filepath = save_chunk_as_npz(chunk_data, save_path)        print(f'Showcase chunk saved: {filepath}')        print()    print('=' * 80)    print('SHOWCASE COMPLETE')    print('=' * 80)    print()    print('Summary:')    print(f'  Processed {len(showcase_results)} camera orientations')    last_chunk = showcase_results[-1]    print(f'  Each NPZ chunk contains:')    print(f'    - {len(last_chunk["roi_data"])} ROIs (24x24 boxes)')    print(f'    - PPG values (shape: {last_chunk["ppg_values"].shape})')    print(f'    - Time deltas (shape: {last_chunk["time_deltas"].shape})')    print(f'    - Landmarks (shape: {last_chunk["landmarks"].shape})')    print(f'    - Metadata and vital signs')    print(f'  Saved to: {os.path.join(OUTPUT_PATH, "showcase")}')    print()    print('Use these NPZ files for:')    print('  - Training models like SCNN_8rois.py')    print('  - Testing preprocessing pipeline')    print('  - Visualizing ROI extraction results')else:    print('No showcase results to visualize')

## 8. Inference Function for Real Videos

In [ ]:
def chunk_video_for_inference(video_path, chunk_size=450, overlap_size=150):
    """
    Chunk a real video for inference using the same strategy as preprocessing.
    
    This function can be reused for inference on new videos.
    
    Args:
        video_path: Path to the video file
        chunk_size: Number of frames per chunk
        overlap_size: Number of overlapping frames
    
    Returns:
        List of video chunks (each as numpy array of shape (chunk_size, H, W, 3))
    """
    # Count frames
    n_frames = count_video_frames_ffmpeg(video_path)
    print(f'Video has {n_frames} frames')
    
    # Create chunks
    chunks = create_chunks(n_frames, chunk_size, overlap_size)
    print(f'Created {len(chunks)} chunks')
    
    # Load and process each chunk
    video_chunks = []
    
    for start, end, chunk_idx in chunks:
        chunk_frames = load_video_frames(video_path, start, end)
        video_chunks.append(chunk_frames)
        print(f'  Chunk {chunk_idx}: {len(chunk_frames)} frames')
    
    return video_chunks

def process_inference_chunk(chunk_frames, detector):
    """
    Process a chunk of frames for inference: detect landmarks and extract ROIs.
    
    Args:
        chunk_frames: Video chunk as numpy array (T, H, W, 3)
        detector: MediaPipeLandmarkDetector instance
    
    Returns:
        Dictionary with ROI data and landmarks
    """
    detector.reset()
    
    # Detect landmarks
    chunk_landmarks = []
    for frame in chunk_frames:
        lms = detector.detect_landmarks(frame)
        if lms is not None:
            chunk_landmarks.append(lms)
        else:
            if chunk_landmarks:
                chunk_landmarks.append(chunk_landmarks[-1].copy())
            else:
                raise RuntimeError('No landmarks detected in first frame')
    
    chunk_landmarks = np.array(chunk_landmarks)
    
    # Extract ROIs
    roi_data = {}
    for roi_name, roi_indices in ROIS.items():
        roi_frames = []
        for frame_idx, frame in enumerate(chunk_frames):
            landmarks = chunk_landmarks[frame_idx]
            bbox = extract_roi_bbox(landmarks, roi_indices, frame.shape[:2], ROI_BOX_SIZE)
            roi_region = extract_roi_region(frame, bbox)
            roi_frames.append(roi_region)
        roi_data[roi_name] = np.array(roi_frames)
    
    return {
        'roi_data': roi_data,
        'landmarks': chunk_landmarks
    }

# Example usage for inference
if MEDIAPIPE_AVAILABLE:
    print('=' * 80)
    print('INFERENCE FUNCTION DEMONSTRATION')
    print('=' * 80)
    print('The chunk_video_for_inference() and process_inference_chunk() functions')
    print('can be used for inference on real videos.')
    print()
    print('Example usage:')
    print('# detector = MediaPipeLandmarkDetector(MEDIAPIPE_MODEL_PATH)')
    print('# video_chunks = chunk_video_for_inference("real_video.mp4")')
    print('# for chunk in video_chunks:')
    print('#     processed = process_inference_chunk(chunk, detector)')
    print('#     # Use processed["roi_data"] for model inference')
    print()
    print('Inference functions ready for use!')
else:
    print('MediaPipe not available - inference functions not demonstrated')

## 9. Summary and Next Steps

In [ ]:
print('=' * 80)
print('SUMMARY AND NEXT STEPS')
print('=' * 80)

print('COMPLETED:')
print('  1. Video frame analysis and PPG sync verification')
print('  2. MediaPipe landmark detection with temporal smoothing')
print('  3. Chunking strategy implementation (450 frames, 150 overlap)')
print('  4. ROI extraction with fixed 24x24 bounding boxes')
print('  5. Complete preprocessing pipeline')
print('  6. Showcase visualization for different camera orientations')
print('  7. Inference functions for real videos')

print('KEY FEATURES:')
print(f'  - Chunk size: {CHUNK_SIZE} frames')
print(f'  - Overlap: {OVERLAP_SIZE} frames')
print(f'  - ROIs: {list(ROIS.keys())}')
print(f'  - ROI box size: {ROI_BOX_SIZE}')
print(f'  - Temporal smoothing window: {SMOOTHING_WINDOW}')
print(f'  - Output format: NPZ with ROIs, PPG values, time deltas, vital signs')

print('NEXT STEPS:')
print('  1. Run full dataset processing (uncomment and modify section 8)')
print('  2. Use saved NPZ files for training models like SCNN_8rois.py')
print('  3. For inference on real videos:')
print('     - Use chunk_video_for_inference() to split video into chunks')
print('     - Use process_inference_chunk() to extract ROIs and landmarks')
print('     - Feed ROI data to trained models')

print('JITTER REDUCTION:')
print('  - Temporal smoothing applied via moving average window')
print('  - Prevents frame-to-frame shivering in ROI masks')
print('  - Reduces non-physiological noise in extracted signals')

print('DATA ALIGNMENT:')
print('  - Frame N corresponds to PPG sync file row N')
print('  - Each chunk maintains exact frame-PPG alignment')
print('  - Overlapping frames ensure continuity between chunks')

print('=' * 80)
print('NOTES:')
print('=' * 80)
print('- For .avi files with encoding issues, we use ffmpeg via imageio')
print('- MediaPipe model path: /home/cristic/face_landmarker.task')
print('- Output directory: /home/cristic/preprocessed_data')
print('- NPZ files contain all data needed for training and inference')
print()
print('Notebook complete! Ready for dataset preprocessing and model training.')